# Setup
## Dependencies

In [25]:
from pathlib import Path
import re
import sqlite3

## File Locations

In [26]:
mosaic_path = Path("../output mosaics/non-flats")
flat_mosaic_path = Path("../output mosaics/flats")

filename_list = sorted(mosaic_path.glob("*-mosaic.fits"))
flat_filename_list = sorted(flat_mosaic_path.glob("*-mosaic.fits"))

master_db_path = Path("../databases/Master_Database.db").resolve()
master_conn = sqlite3.connect(master_db_path)

## Functions

In [27]:
def get_group_id(filename_list):
    mosaic_records = []

    for files in filename_list:
        filename = files.name
        match = re.search(r"-g(\d+)-mosaic\.fits$", filename)

        if match is None:
            print("no ID, skipping:", filename)
            continue

        group_id = int(match.group(1))
        mosaic_records.append((group_id, filename))

    return(mosaic_records)

In [28]:
def insert_mosaic_names(master_conn, mosaic_records):
    
    sql = """
        UPDATE observations
        SET mosaic_name = ?
        WHERE group_id = ?
          AND mosaic_name IS NULL
    """
    updated = 0
    with master_conn:
        for group_id, filename in mosaic_records:
            cursor = master_conn.execute(sql, (filename, group_id))
            updated += cursor.rowcount
    return updated

In [29]:
def insert_flat_mosaic_names(master_conn, mosaic_records):
    sql = """
        UPDATE observations
        SET mosaic_name_f = ?
        WHERE group_id = ?
          AND mosaic_name_f IS NULL
    """
    updated = 0
    with master_conn:
        for group_id, filename in mosaic_records:
            cursor = master_conn.execute(sql, (filename, group_id))
            updated += cursor.rowcount
    return updated

# Actual Code Execution

In [30]:
mosaic_records = get_group_id(filename_list)
num_done = insert_mosaic_names(master_conn, mosaic_records)

flat_mosaic_records = get_group_id(flat_filename_list)
flat_num_done = insert_flat_mosaic_names(master_conn, flat_mosaic_records)
print(f"Non flat: {num_done}")
print(f"With flat: {flat_num_done}")


no ID, skipping: sn2026aeiv-26_10_16-east-16-r-2-mosaic.fits
no ID, skipping: sn2026aeiv-26_10_16-east-16-r-mosaic.fits
Non flat: 0
With flat: 0
